In [14]:
import csv
import json
import random
import sys
import pandas as pd
from collections import Counter, defaultdict
from pathlib import Path

ROOT = Path.cwd().parent
SRC = ROOT/"Dataset"/"issues_test.csv"
OUT = ROOT/"Dataset"/"eval.jsonl"


In [15]:
df = pd.read_csv(SRC)
print(len(df))

1500


In [18]:
df.columns

Index(['repo', 'created_at', 'label', 'title', 'body'], dtype='str')

In [19]:
df.insert(0, 'Serial Number', range(1, len(df) + 1))
df.columns

Index(['Serial Number', 'repo', 'created_at', 'label', 'title', 'body'], dtype='str')

In [24]:
grouped_row_count = df.groupby("label").size()
grouped_row_count

label
bug         500
feature     500
question    500
dtype: int64

In [37]:
df = df.fillna("")
sample = df.groupby("label").sample(n=100, random_state=42)
print(len(sample))
print(sample["label"].value_counts())
print(sample["repo"].value_counts())


300
label
bug         100
feature     100
question    100
Name: count, dtype: int64
repo
facebook/react           66
bitcoin/bitcoin          64
microsoft/vscode         59
opencv/opencv            57
tensorflow/tensorflow    54
Name: count, dtype: int64


In [38]:
with open(OUT, "w", encoding="utf-8") as f:
    for i, row in sample.iterrows():
        case = {
            "id": int(i),   # original row number; int() because JSON can't store numpy numbers
            "repo": row["repo"],
            "expected": row["label"],
            "title": row["title"].strip(),
            "body": row["body"].replace("\r\n", "\n").strip()[:1500],
        }
        f.write(json.dumps(case, ensure_ascii=False) + "\n")